# 07 — Wealth Growth & 10-Year Net Worth Projection

This notebook tracks your portfolio's growth over time and projects net worth toward financial independence.

- **Growth chart** — month-over-month portfolio value from historical snapshots
- **Growth metrics** — MoM return, annualized return (CAGR)
- **10-year projection** — compound growth with monthly contributions and income growth
- **Milestone tracker** — when you'll hit $250K, $500K, $1M

**Prerequisites:**
- Run notebook 02 at least once (creates `portfolio_history.json`)
- `portfolio_snapshot.json` must exist for current portfolio value
- Run `uv sync` to install matplotlib

## Your Financial Profile

Adjust these variables to match your situation. Re-run the notebook after changes.

In [ ]:
# === YOUR FINANCIAL PROFILE ===
CURRENT_AGE = 31
TARGET_FI_AGE = 45                       # Financial independence target age
MONTHLY_CONTRIBUTION = 1000              # $ invested per month across all brokerages
ANNUAL_CONTRIBUTION_INCREASE = 5.0       # % yearly increase in contributions (income growth)
ASSUMED_ANNUAL_RETURN = 8.0              # % — used when not enough history to calculate
# ==============================

PROJECTION_YEARS = TARGET_FI_AGE - CURRENT_AGE
print(f"Profile: age {CURRENT_AGE}, targeting FI at {TARGET_FI_AGE} ({PROJECTION_YEARS} years)")
print(f"Monthly contribution: ${MONTHLY_CONTRIBUTION:,} (growing {ANNUAL_CONTRIBUTION_INCREASE}%/yr)")
print(f"Assumed return: {ASSUMED_ANNUAL_RETURN}% annual (until enough history to calculate)")

In [ ]:
import json
import os
from datetime import datetime, timedelta

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

# Dark theme to match project aesthetic
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor": "#1a1a2e",
    "axes.facecolor": "#16213e",
    "axes.edgecolor": "#333",
    "axes.labelcolor": "#ccc",
    "xtick.color": "#888",
    "ytick.color": "#888",
    "grid.color": "#2a2a4a",
    "text.color": "#e0e0e0",
    "figure.figsize": (12, 6),
})

# Load portfolio history
HISTORY_FILE = os.path.join("..", "portfolio_history.json")
SNAPSHOT_FILE = os.path.join("..", "portfolio_snapshot.json")

if os.path.exists(HISTORY_FILE):
    with open(HISTORY_FILE, "r") as f:
        history = json.load(f)
else:
    history = []

if not os.path.exists(SNAPSHOT_FILE):
    raise FileNotFoundError("portfolio_snapshot.json not found. Run notebook 02 first.")

with open(SNAPSHOT_FILE, "r") as f:
    snapshot = json.load(f)

current_value = snapshot["summary"]["total_value"]
current_cost = snapshot["summary"]["total_cost_basis"]

print(f"\u2705 Current portfolio: ${current_value:,.2f}")
print(f"\u2705 History entries: {len(history)}")

if history:
    hist_df = pd.DataFrame(history)
    hist_df["date"] = pd.to_datetime(hist_df["date"])
    hist_df = hist_df.sort_values("date").reset_index(drop=True)
    print(f"   Date range: {hist_df['date'].iloc[0].strftime('%Y-%m-%d')} to {hist_df['date'].iloc[-1].strftime('%Y-%m-%d')}")

## Portfolio Growth History

In [ ]:
if not history:
    print("No history yet. Run notebook 02 to record your first snapshot.")
    print("Growth chart will appear after your first pipeline run.")

elif len(hist_df) == 1:
    # Single data point — show as a dot
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(hist_df["date"], hist_df["total_value"], "o", color="#22c55e", markersize=12)
    ax.annotate(
        f"${hist_df['total_value'].iloc[0]:,.0f}\nHistory starts here",
        xy=(hist_df["date"].iloc[0], hist_df["total_value"].iloc[0]),
        xytext=(20, 20), textcoords="offset points",
        fontsize=11, color="#22c55e",
        arrowprops=dict(arrowstyle="->", color="#22c55e"),
    )
    ax.set_title("Portfolio Value Over Time", fontsize=14, fontweight="bold")
    ax.set_ylabel("Portfolio Value ($)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\nOnly 1 snapshot so far. Run the pipeline over the coming months to build history.")

else:
    # Multiple data points — full chart
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(hist_df["date"], hist_df["total_value"], "-o", color="#22c55e",
            linewidth=2, markersize=6, label="Portfolio Value")
    ax.plot(hist_df["date"], hist_df["total_cost_basis"], "--", color="#666",
            linewidth=1.5, label="Cost Basis")

    # Fill between value and cost basis
    ax.fill_between(
        hist_df["date"], hist_df["total_cost_basis"], hist_df["total_value"],
        where=hist_df["total_value"] >= hist_df["total_cost_basis"],
        color="#22c55e", alpha=0.15, label="Gain",
    )
    ax.fill_between(
        hist_df["date"], hist_df["total_cost_basis"], hist_df["total_value"],
        where=hist_df["total_value"] < hist_df["total_cost_basis"],
        color="#ef4444", alpha=0.15, label="Loss",
    )

    # Annotate latest value
    latest = hist_df.iloc[-1]
    ax.annotate(
        f"${latest['total_value']:,.0f}",
        xy=(latest["date"], latest["total_value"]),
        xytext=(10, 10), textcoords="offset points",
        fontsize=11, fontweight="bold", color="#22c55e",
    )

    ax.set_title("Portfolio Value Over Time", fontsize=14, fontweight="bold")
    ax.set_ylabel("Portfolio Value ($)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.legend(loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Month-over-month change table
    print("\n" + "=" * 60)
    print("  MONTH-OVER-MONTH CHANGES")
    print("=" * 60)
    for i in range(1, len(hist_df)):
        prev = hist_df.iloc[i - 1]
        curr = hist_df.iloc[i]
        change = curr["total_value"] - prev["total_value"]
        change_pct = (change / prev["total_value"] * 100) if prev["total_value"] > 0 else 0
        emoji = "\U0001f7e2" if change >= 0 else "\U0001f534"
        print(f"  {prev['date'].strftime('%b %d')} -> {curr['date'].strftime('%b %d')}:  "
              f"${curr['total_value']:>12,.2f}  ({emoji} {change:+,.2f} / {change_pct:+.2f}%)")

## Growth Metrics

In [ ]:
effective_annual_return = ASSUMED_ANNUAL_RETURN  # default

print("=" * 60)
print("         GROWTH METRICS")
print("=" * 60)

if history and len(hist_df) >= 2:
    first = hist_df.iloc[0]
    last = hist_df.iloc[-1]
    days_elapsed = (last["date"] - first["date"]).days

    total_return_pct = (last["total_value"] / first["total_value"] - 1) * 100
    print(f"  Total return over period:  {total_return_pct:+.2f}%")
    print(f"  Period: {days_elapsed} days ({first['date'].strftime('%Y-%m-%d')} to {last['date'].strftime('%Y-%m-%d')})")

    if days_elapsed >= 90:  # 3+ months — use CAGR
        years_elapsed = days_elapsed / 365.25
        cagr = ((last["total_value"] / first["total_value"]) ** (1 / years_elapsed) - 1) * 100
        effective_annual_return = max(cagr, 0)  # floor at 0%
        print(f"  Annualized return (CAGR): {cagr:+.2f}%")
        print(f"\n  Using calculated CAGR ({effective_annual_return:.1f}%) for projection.")
    else:
        print(f"\n  Not enough history for reliable CAGR (need 3+ months, have {days_elapsed} days).")
        print(f"  Using assumed return ({ASSUMED_ANNUAL_RETURN}%) for projection.")
else:
    print(f"  Not enough data points for growth metrics (have {len(history)}, need 2+).")
    print(f"  Using assumed return ({ASSUMED_ANNUAL_RETURN}%) for projection.")
    print(f"\n  Run the pipeline again next month to start building history.")

print(f"\n  Effective annual return for projection: {effective_annual_return:.1f}%")
print("=" * 60)

## 10-Year Net Worth Projection

Projects portfolio growth from current value using compound returns and monthly contributions.
Contributions increase annually to account for expected income growth.

In [ ]:
# Month-by-month projection
monthly_rate = effective_annual_return / 100 / 12
projection_months = PROJECTION_YEARS * 12

rows = []
balance = current_value
cumulative_contributions = 0.0
starting_principal = current_value
monthly_contrib = float(MONTHLY_CONTRIBUTION)
current_year = datetime.now().year

# Record month 0 (today)
rows.append({
    "month": 0,
    "year": current_year,
    "age": CURRENT_AGE,
    "portfolio_value": balance,
    "cumulative_contributions": 0.0,
    "cumulative_growth": 0.0,
    "principal": starting_principal,
})

for m in range(1, projection_months + 1):
    # Increase contributions at the start of each new year
    if m > 1 and m % 12 == 1:
        monthly_contrib *= (1 + ANNUAL_CONTRIBUTION_INCREASE / 100)

    # Compound growth + contribution
    growth = balance * monthly_rate
    balance = balance + growth + monthly_contrib
    cumulative_contributions += monthly_contrib

    year = current_year + (m // 12) if m % 12 != 0 else current_year + (m // 12) - 1
    age = CURRENT_AGE + m / 12

    rows.append({
        "month": m,
        "year": current_year + ((m - 1) // 12),
        "age": CURRENT_AGE + m / 12,
        "portfolio_value": balance,
        "cumulative_contributions": cumulative_contributions,
        "cumulative_growth": balance - starting_principal - cumulative_contributions,
        "principal": starting_principal,
    })

proj_df = pd.DataFrame(rows)

final = proj_df.iloc[-1]
print(f"Projected portfolio at age {TARGET_FI_AGE}: ${final['portfolio_value']:,.0f}")
print(f"  Starting principal:       ${starting_principal:,.0f}")
print(f"  Total contributions:      ${final['cumulative_contributions']:,.0f}")
print(f"  Investment growth:        ${final['cumulative_growth']:,.0f}")
print(f"  Growth rate used:         {effective_annual_return:.1f}% annual")

In [ ]:
# Projection chart — stacked area
fig, ax = plt.subplots(figsize=(14, 7))

months = proj_df["month"]
ages = proj_df["age"]

# Stacked areas
ax.fill_between(months, 0, proj_df["principal"],
                color="#2563eb", alpha=0.6, label=f"Starting Principal (${starting_principal:,.0f})")
ax.fill_between(months, proj_df["principal"],
                proj_df["principal"] + proj_df["cumulative_contributions"],
                color="#22c55e", alpha=0.6, label="Cumulative Contributions")
ax.fill_between(months, proj_df["principal"] + proj_df["cumulative_contributions"],
                proj_df["portfolio_value"],
                color="#f59e0b", alpha=0.6, label="Investment Growth")

# Portfolio value line on top
ax.plot(months, proj_df["portfolio_value"], color="#fff", linewidth=2, alpha=0.8)

# Milestone lines
milestones = [250_000, 500_000, 1_000_000]
milestone_colors = ["#22c55e", "#eab308", "#ef4444"]
max_val = proj_df["portfolio_value"].max()
for target, color in zip(milestones, milestone_colors):
    if target <= max_val * 1.1:  # only show if within range
        ax.axhline(y=target, color=color, linestyle="--", alpha=0.5, linewidth=1)
        ax.text(projection_months * 0.02, target * 1.02,
                f"${target / 1000:.0f}K" if target < 1_000_000 else "$1M",
                color=color, fontsize=10, alpha=0.7)

# Annotate final value
ax.annotate(
    f"${final['portfolio_value']:,.0f}\nAge {TARGET_FI_AGE}",
    xy=(projection_months, final["portfolio_value"]),
    xytext=(-80, 15), textcoords="offset points",
    fontsize=12, fontweight="bold", color="#fff",
    arrowprops=dict(arrowstyle="->", color="#fff"),
)

# Formatting
ax.set_title(f"Net Worth Projection: Age {CURRENT_AGE} to {TARGET_FI_AGE} ({effective_annual_return:.1f}% annual return)",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Years from Now")
ax.set_ylabel("Portfolio Value ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# X axis: show years instead of months
year_ticks = list(range(0, projection_months + 1, 12))
year_labels = [f"Yr {y // 12}\n(Age {CURRENT_AGE + y // 12})" for y in year_ticks]
ax.set_xticks(year_ticks)
ax.set_xticklabels(year_labels, fontsize=8)

ax.legend(loc="upper left", fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Year-by-Year Breakdown

In [ ]:
# Extract year-end snapshots
yearly = proj_df[proj_df["month"] % 12 == 0].copy()
# Also include the final month if it's not a year boundary
if proj_df.iloc[-1]["month"] % 12 != 0:
    yearly = pd.concat([yearly, proj_df.iloc[[-1]]])

print("=" * 90)
print(f"{'Year':>6} {'Age':>5} {'Start Value':>14} {'Contributions':>14} {'Growth':>14} {'End Value':>14}")
print("=" * 90)

prev_value = starting_principal
prev_contrib = 0.0

for _, row in yearly.iterrows():
    if row["month"] == 0:
        continue

    year_contrib = row["cumulative_contributions"] - prev_contrib
    year_growth = row["portfolio_value"] - prev_value - year_contrib
    age = int(CURRENT_AGE + row["month"] / 12)
    year = int(row["year"])

    print(f"{year:>6} {age:>5} ${prev_value:>12,.0f} ${year_contrib:>12,.0f} ${year_growth:>12,.0f} ${row['portfolio_value']:>12,.0f}")

    prev_value = row["portfolio_value"]
    prev_contrib = row["cumulative_contributions"]

print("=" * 90)
print(f"\n  Total contributions over {PROJECTION_YEARS} years: ${final['cumulative_contributions']:,.0f}")
print(f"  Total investment growth:                ${final['cumulative_growth']:,.0f}")
print(f"  Final projected value:                  ${final['portfolio_value']:,.0f}")

## Milestone Tracker

In [ ]:
print("=" * 60)
print("         MILESTONE TRACKER")
print("=" * 60)

milestones = [250_000, 500_000, 750_000, 1_000_000, 1_500_000, 2_000_000]

for target in milestones:
    target_label = f"${target:,.0f}"

    if current_value >= target:
        print(f"  {target_label:>12}: Already reached! Current value: ${current_value:,.0f}")
        continue

    hit = proj_df[proj_df["portfolio_value"] >= target]
    if not hit.empty:
        first_hit = hit.iloc[0]
        months_away = int(first_hit["month"])
        age_at = CURRENT_AGE + months_away / 12
        year_at = datetime.now().year + months_away // 12
        print(f"  {target_label:>12}: Age {age_at:.0f} (month {months_away}, ~{year_at})")
    else:
        print(f"  {target_label:>12}: Not reached within {PROJECTION_YEARS}-year projection")

print("=" * 60)

---

### Assumptions & Disclaimer

**Assumptions used in this projection:**
- Annual return rate is applied as a constant monthly rate (real markets are volatile)
- Monthly contributions increase by a fixed percentage annually (income growth)
- No taxes, withdrawals, or major life events factored in
- This projection covers this brokerage portfolio only (not 401k, real estate, etc.)
- As real history builds up (3+ months), the assumed return rate will be replaced by your actual CAGR

**Re-run this notebook periodically** to update the projection with real growth data.

**This is not financial advice.** Projections are illustrative. Past performance does not guarantee future results.